# Tutorial 04: NVF (Normal Vector Field) Fourier Factorization

This tutorial demonstrates **Normal Vector Field (NVF)** 2D Fourier factorization in
`wavebit` for curved, high-contrast structures (here a silicon circular pillar, n=3.5),
where the staircase approximation of a plain discrete Fourier sum converges poorly.

### Contents
1. **Geometry and its normal vector field**
2. **Convergence: the sampled path vs NVF across Fourier harmonics**
3. **Why a signed-distance field, and not a plain gradient?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import wavebit

## 1. Silicon circular pillar (n=3.5) and its normal vector field

The normal vector field says which way each material interface points. It is a property of
the *geometry*, not of the permittivity riding on it, so `wavebit` computes it once from a
signed-distance field and then holds it fixed. `freeze_geometry()` is the public way to take
that decision and look at it; the same object is what keeps a solve traceable under
`jax.jit`.

It is a **unit** vector across the whole cell, not only on the boundary. Away from any
interface its *direction* carries no physical meaning — but its *magnitude* must still be 1,
and section 3 shows why the factorization falls apart otherwise.

In [ ]:
# 100x100 grid, high-contrast silicon pillar
N = 100
grid_y, grid_x = np.ogrid[:N, :N]
radius = 30
dist = np.sqrt((grid_x - N // 2) ** 2 + (grid_y - N // 2) ** 2)

eps_matrix = np.ones((N, N), dtype=complex)
eps_matrix[dist <= radius] = 3.5 ** 2  # silicon pillar (n = 3.5, so eps = 12.25)

# The raster IS the permittivity map -- there is no conversion on the way in. Before the
# material convention changed this line took a square root, handed the solver an index,
# and the solver squared it back.
structure_raster = np.expand_dims(eps_matrix, axis=0)  # relative permittivity, 1 layer

option_common = dict(
    psi=0.0, top=1.0, bot=2.1025,          # half-space permittivities: vacuum, n = 1.45 glass
    theta=15.0 * np.pi / 180, phi=30.0 * np.pi / 180,
    wavelength=1550.0, period=dict(x=1000, y=1000), thickness=[500],
    structure=structure_raster,
)

# The normal field is taken on the raster as given, lifted only to the
# 4*truncation_order+1-point floor its transform needs. See docs/nvf-upsampling.md.
mee_nvf = wavebit.rcwa(backend=0, factorization='nvf', truncation_order=dict(x=5, y=5),
                            **option_common)
nx, ny = mee_nvf.freeze_geometry().normal_field[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(eps_matrix.real, cmap='viridis', origin='lower')
axes[0].set_title('Dielectric map (eps, n=3.5)')
axes[1].imshow(nx, cmap='coolwarm', origin='lower')
axes[1].set_title('Normal field $n_u$')
axes[2].imshow(ny, cmap='coolwarm', origin='lower')
axes[2].set_title('Normal field $n_y$')
plt.tight_layout()
plt.show()

## 2. Convergence: the sampled path vs NVF

Both runs go through the public API and differ only in `factorization`: `'li'` is the discrete
Fourier sum, `'nvf'` selects the NVF factorization.

In [ ]:
max_order_list = [1, 2, 3, 4, 5, 6, 7, 8]
R_dfs_list, R_nvf_list = [], []

for max_order_val in max_order_list:
    dfs = wavebit.rcwa(backend=0, factorization='li', truncation_order=[max_order_val, max_order_val],
                             **option_common).solve()
    nvf = wavebit.rcwa(backend=0, factorization='nvf', truncation_order=[max_order_val, max_order_val],
                             **option_common).solve()
    R_dfs_list.append(float(dfs.R.sum()))
    R_nvf_list.append(float(nvf.R.sum()))

plt.figure(figsize=(9, 5.5))
plt.plot(max_order_list, R_dfs_list, 'o--', label='Sampled (li)', color='tab:red')
plt.plot(max_order_list, R_nvf_list, 'd-', label='NVF', color='tab:blue', linewidth=2.5)
plt.xlabel('Fourier harmonics (truncation_order)')
plt.ylabel('Reflectance (R)')
plt.title('Convergence: sampled coefficients vs NVF')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Why a signed-distance field, and not a plain gradient?

The obvious way to get a normal field is to take the gradient of `eps` directly — a Sobel
operator, say. `wavebit` does not offer that, and this section shows why.

The NVF factorization builds a projector out of the normal field,

$$P_n = n n^T = \begin{pmatrix} n_u^2 & n_u n_y \\ n_u n_y & n_y^2 \end{pmatrix},
\qquad \epsilon_{\text{eff}} = \langle\epsilon\rangle - \Delta \cdot P_n ,
\qquad \Delta = \langle\epsilon\rangle - \langle 1/\epsilon\rangle^{-1}$$

and that derivation needs $P_n$ to be a genuine projector — which holds only if
$n_u^2 + n_y^2 = 1$ **everywhere in the cell**. In Fourier space that is the identity
$C_{xx} + C_{yy} = I$.

A raw $\nabla\epsilon$ is nonzero only in the one-pixel band at the interface, so it fails
that condition: $P_n \approx 0$ over the bulk, which sends
$\epsilon_{\text{eff}} \to \langle\epsilon\rangle$ — the plain Laurent rule, i.e. NVF
switched off. The signed-distance field instead *extends* the interface normal into the bulk
at unit length. Its direction there is physically meaningless (there is no interface to be
normal to), but its magnitude is what keeps the projector complete.

In [ ]:
from wavebit._core.factorization import (fourier2d_piecewise_constant, fourier2d_pointwise,
                                           normal_vector_field, nvf_conv_matrices)
import array_api_compat.numpy as xnp


def sobel_normal_field(eps):
    """A plain periodic Sobel gradient of eps — the tempting alternative, for comparison."""
    p = np.pad(eps.real, ((1, 1), (1, 1)), mode='wrap')
    gx = (p[1:-1, 2:] - p[1:-1, :-2]) * 2 / 8 + (p[:-2, 2:] - p[:-2, :-2]) / 8 \
        + (p[2:, 2:] - p[2:, :-2]) / 8
    gy = (p[2:, 1:-1] - p[:-2, 1:-1]) * 2 / 8 + (p[2:, :-2] - p[:-2, :-2]) / 8 \
        + (p[2:, 2:] - p[:-2, 2:]) / 8
    norm = np.hypot(gx, gy)
    m = norm > 1e-8
    ux, uy = np.zeros_like(gx), np.zeros_like(gy)
    ux[m], uy[m] = gx[m] / norm[m], gy[m] / norm[m]
    return ux, uy


def projector_defect(ux, uy, truncation_order=3):
    """||C_xx + C_yy - I|| / ||I||  — zero exactly when |n| = 1 over the cell."""
    C_xx = fourier2d_pointwise(xnp, (ux ** 2).astype(complex), truncation_order, truncation_order)
    C_yy = fourier2d_pointwise(xnp, (uy ** 2).astype(complex), truncation_order, truncation_order)
    I = np.eye(C_xx.shape[0])
    return float(np.linalg.norm(C_xx + C_yy - I) / np.linalg.norm(I))


sdf_field = normal_vector_field(eps_matrix)
sobel_field = sobel_normal_field(eps_matrix)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for a, (label, (ux, uy)) in zip(axes[:2], [('SDF (what wavebit uses)', sdf_field),
                                           ('Plain Sobel gradient', sobel_field)]):
    mag = np.hypot(ux, uy)
    im = a.imshow(mag, cmap='magma', vmin=0, vmax=1, origin='lower')
    a.set_title(f'{label}\n$|n|$ = 1 over {np.isclose(mag, 1, atol=1e-8).mean() * 100:.1f}% of the cell')
    a.set_xticks([]); a.set_yticks([])
    plt.colorbar(im, ax=a, fraction=0.046)

row = N // 2
axes[2].plot(np.hypot(*sdf_field)[row], color='tab:blue', lw=2, label='SDF')
axes[2].plot(np.hypot(*sobel_field)[row], color='tab:red', lw=2, label='Sobel')
axes[2].set_title(f'$|n|$ along the centre row (y={row})')
axes[2].set_xlabel('x (pixels)'); axes[2].set_ylim(-0.05, 1.15)
axes[2].legend(loc='center left'); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"projector defect ||C_xx + C_yy - I||/||I||")
print(f"  SDF   : {projector_defect(*sdf_field):.5f}")
print(f"  Sobel : {projector_defect(*sobel_field):.5f}")

The single dip in the SDF curve at the centre is the medial axis of the distance map — the
ridge equidistant from both sides of the pillar, where the gradient vanishes. It is a
lower-dimensional set, so its share of the cell shrinks as the raster is refined, and with it
the defect. The Sobel field has the opposite behaviour: refining the grid makes the interface
band a *smaller* fraction of the cell, so its projector collapses further.

Finally, how much of the available inverse-rule correction each field actually applies —
100% would mean the full $\Delta$, and 0% means NVF has done nothing at all:

In [ ]:
max_order_c = 3

# `fourier2d_piecewise_constant` because eps is a raster: each pixel holds its value across the
# whole cell, so the coefficients have an exact closed form. It matters here beyond accuracy
# -- `nvf_conv_matrices` below builds its own <eps> the same way, so using the sampled build
# would make E a different matrix from the one the comparison is against, and the percentages
# would carry a pixel-correction difference that has nothing to do with the normal field.
# (The projector above is the opposite case, and calls `fourier2d_pointwise`. See
# docs/factorization-terms.md.)
E = fourier2d_piecewise_constant(xnp, eps_matrix, max_order_c, max_order_c, inverse_rule=None)       # <eps>, direct rule
A = np.linalg.inv(
    fourier2d_piecewise_constant(xnp, 1 / eps_matrix, max_order_c, max_order_c, inverse_rule=None))  # <1/eps>^-1
available = np.linalg.norm(E - A)

for label, (ux, uy) in [('SDF', sdf_field), ('Sobel', sobel_field)]:
    eps_xx = nvf_conv_matrices(xnp, eps_matrix, ux, uy, max_order_c, max_order_c)[0]
    print(f'{label:6s}: {np.linalg.norm(eps_xx - E) / available * 100:5.1f}% of the correction applied')

So the Sobel field pays the full cost of the NVF path — five convolution matrices, an extra
inversion, the anisotropic 2D solve — and gets back something within a couple of percent of
plain Laurent. That is why it is not offered as a method.

Note that energy conservation does *not* catch this: the Hermitian symmetrization in
`nvf_conv_matrices` guarantees real $q^2$ regardless of whether the projector is valid, so
both fields conserve energy to ~1e-11. The invariant above is what actually distinguishes
them, and it is pinned down in `tests/test_nvf_normal_field.py`.